In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Object-Oriented Design for Implementation

The function `train` of that section trained linear
regression in a dozen lines. Those lines are written for one model and one
dataset: the functions `linreg` and `squared_loss`, the parameters `w` and
`b`, and the arrays `X_train` and `X_val` all appear in its body, so any
other model would need a copy with a few lines changed. Stripped of these
names, the loop reads as follows.

```text
for epoch in range(num_epochs):
    for X, y in train_data:                     # data
        loss = loss_fn(net(X), y)               # model and loss
        update(params, gradient(loss, params))  # optimizer
    val_loss = mean_loss(net, val_data)         # model on held-out data
    record(epoch, val_loss)
```

Only the lines with comments depend on the problem; the loop around them
never changes. Nearly thirty sections of this book train a model, most of
them in several frameworks, so a loop written out in each would exist in
close to a hundred copies. A change to how we train, such as clipping very
large gradients or lowering the learning rate as training proceeds, would
then have to be made in every copy, and the copies would drift apart.

We therefore write the loop once, as a class, and put the parts that change
into two more classes, a design borrowed from libraries such as
[Lightning](https://lightning.ai/). A **`Module`** holds a model: its
parameters, its predictions (`forward`), its loss, and its *optimizer*, the
object that applies an update rule such as minibatch SGD. A
**`DataModule`** holds a dataset and serves it in minibatches. A
**`Trainer`** runs the loop.

In the pseudocode, the `DataModule` supplies
`train_data` and `val_data`, the `Module` supplies `net`, `loss_fn`, and the
update rule, and the `Trainer` runs the two `for` loops, computes the
gradients, and records the losses (the figure). Every model in
the book is a subclass of `Module`, and every dataset a subclass of
`DataModule`.

![The three base classes. The `Trainer` takes minibatches from a `DataModule` and passes each to a `Module`, whose `training_step` returns the loss; the trainer updates the parameters and keeps the epoch means of the losses in its `history`. New models and datasets are written as subclasses of `Module` and `DataModule`.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-linreg-oo-classes.svg)

This section rewrites the code of that section in this
form and ends by training the same model once more, through the classes.
Some arguments and methods exist only for later chapters; we say so where
they appear.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
import collections

## Utilities

Two helpers make the classes of this book easier to write in a notebook.
Neither has anything to do with machine learning. Python defines a class in
one block of code, but a class with several methods is easier to explain in
several short cells, each next to the prose about it. The decorator
`add_to_class` registers a function as a method of a class that already
exists, so that one class can be spread over several cells. This is
notebook scaffolding, not a design we recommend: in an ordinary Python
module, a class is defined in one place.

In [ ]:
def add_to_class(Class):
    """Register functions as methods in created class."""
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
        return obj
    return wrapper

To see it at work, we create an instance `a` of a class `A` that has no
methods other than its constructor.

In [ ]:
class A:
    def __init__(self):
        self.b = 1

a = A()

In a separate cell we register a method `do`. The instance `a`, created
before `do` existed, has it too, since Python looks up a method on the class
at the moment of the call.

In [ ]:
@add_to_class(A)
def do(self):
    print('Instance attribute "b" is', self.b)

a.do()

The second helper removes repetitive code from constructors, which usually
store each argument as an attribute: `self.lr = lr`,
`self.batch_size = batch_size`, and so on. Most of these arguments are
hyperparameters, such as the learning rate, hence the name of the `d2l`
class `HyperParameters`. Its method `save_hyperparameters` stores all
arguments of the constructor that calls it as attributes, except those named
in `ignore`. It reads them with Python's `inspect` module, which teaches
nothing about training, so its body is in that section.

In [ ]:
class B(d2l.HyperParameters):
    def __init__(self, a, b, c):
        self.save_hyperparameters(ignore=['c'])
        print('self.a =', self.a, 'self.b =', self.b)
        print('self.c exists:', hasattr(self, 'c'))

b = B(a=1, b=2, c=3)

All classes in the rest of this section inherit from `HyperParameters`.

## Models

A model in this book is a subclass of `Module`. The subclass creates its
parameters in its constructor and defines three methods: `forward` computes
the predictions for a minibatch of inputs, `loss` compares predictions with
labels, and `configure_optimizers` returns the optimizer. (By default,
`forward` applies `self.net`, a network that a subclass may store.) The base
class contributes the two steps that the trainer calls, `training_step` and
`validation_step`. Besides `HyperParameters`, `Module` inherits from the
framework's base class of networks.

That class is `tf.keras.Model`, whose instances run `call` when called; our
`call` stores the flag `training`, for layers such as dropout, and
redirects to `forward`. The constructor removes an attribute `loss` that
Keras 3 sets to `None`, since it would hide our method `loss`.

In [ ]:
class Module(tf.keras.Model, d2l.HyperParameters):
    """The base class of models."""
    def __init__(self):
        super().__init__()
        self.training = None
        self.__dict__.pop('loss', None)

    def loss(self, y_hat, y):
        raise NotImplementedError

    def forward(self, X):
        assert hasattr(self, 'net'), 'Neural network is not defined'
        return self.net(X)

    def call(self, X, *args, training=None, **kwargs):
        if training is not None:
            self.training = training
        return self.forward(X, *args, **kwargs)

    def training_step(self, batch):
        """The mean loss on one minibatch, or a dict of batch-mean metrics
        with key 'loss'."""
        return self.loss(self(*batch[:-1], training=True), batch[-1])

    def validation_step(self, batch):
        """The mean loss on one minibatch of validation data, or a dict of
        batch-mean metrics with key 'loss'."""
        return self.loss(self(*batch[:-1], training=False), batch[-1])

    def configure_optimizers(self):
        raise NotImplementedError

A minibatch is a tuple whose last element holds the labels and whose other
elements hold the inputs. `training_step` calls the model on the inputs and
passes the predictions and the labels to `loss`; `validation_step` does the
same on validation data. Both steps *return* their result and do nothing
else: they do not print, plot, or keep a record. This restriction is the
key decision of the design. Nothing in a model refers to the trainer, so any
loop, not only the trainer's, can train or evaluate it.

In this chapter a step returns one number, the mean loss over the minibatch.
From that section on, a step may return a dictionary of such
means, with the loss under the key `'loss'` and, say, the accuracy next to
it. The function `as_metrics` wraps a single loss in a dictionary, so that
the trainer handles both alike.

In [ ]:
def as_metrics(out):
    """A step returns a loss or a dict of metrics; return a dict."""
    return out if isinstance(out, dict) else {'loss': out}

### Linear Regression as a Module

The model of that section becomes a subclass of `Module`.
Its constructor creates the parameters as `init_params` did and registers
them with the module through the framework's parameter type. Registration
lets the framework's tools find the parameters, which are then listed,
saved, and moved to a GPU together with the rest of the model. It computes
nothing for us: the loss and the update are still the ones we wrote by hand.

`add_weight` creates a variable, a tensor that can be changed in place, and
registers it with the Keras model, so `model.trainable_variables` lists `w`
and `b`.

In [ ]:
class LinearRegressionScratch(d2l.Module):
    """The linear regression model implemented from scratch."""
    def __init__(self, num_inputs, lr, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        normal = tf.keras.initializers.RandomNormal(mean=0, stddev=sigma)
        self.w = self.add_weight(name='w', shape=(num_inputs, 1),
                                 initializer=normal)
        self.b = self.add_weight(name='b', shape=(1,), initializer='zeros')

The forward pass and the loss are `linreg` and `squared_loss` of
that section, now methods that read the parameters from the
model.

In [ ]:
@d2l.add_to_class(LinearRegressionScratch)
def forward(self, X):
    return d2l.matmul(X, self.w) + self.b

In [ ]:
@d2l.add_to_class(LinearRegressionScratch)
def loss(self, y_hat, y):
    l = (y_hat - d2l.reshape(y, (-1, 1))) ** 2 / 2
    return d2l.reduce_mean(l)

The update becomes an optimizer class, `SGD`. Its update is the one line of
that section; the rest of the class gives it the interface
of the framework's own optimizers, so that that section can
exchange the two.

As in Keras optimizers, `apply_gradients` receives pairs of a gradient and a
variable and updates each variable in place with `assign_sub`.

In [ ]:
class SGD(d2l.HyperParameters):
    """Minibatch stochastic gradient descent."""
    def __init__(self, lr):
        self.save_hyperparameters()

    def apply_gradients(self, grads_and_vars):
        for grad, param in grads_and_vars:
            param.assign_sub(self.lr * grad)

The model's `configure_optimizers` returns an instance of this class.

In [ ]:
@d2l.add_to_class(LinearRegressionScratch)
def configure_optimizers(self):
    return SGD(self.lr)

## Data

The base class for data is shorter still. A dataset in this book is a
subclass of `DataModule`, whose constructor prepares the data, downloading
and preprocessing it if needed. (The argument `root` names the directory for
downloaded files, and `num_workers` the number of processes that load data
in parallel.) A subclass implements one method, `get_dataloader(train)`. It
returns a *data loader*, an object that yields the data one minibatch at a
time, as the tuples that the steps of `Module` expect. The other two methods
merely call it.

In [ ]:
class DataModule(d2l.HyperParameters):
    """The base class of data."""
    def __init__(self, root='../data', num_workers=4):
        self.save_hyperparameters()

    def get_dataloader(self, train):
        raise NotImplementedError

    def train_dataloader(self):
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        return self.get_dataloader(train=False)

The synthetic data of that section become a
subclass whose constructor calls `d2l.synthetic_data` for
`num_train + num_val` examples, by default 1,000 of each: the first
`num_train` for training, the rest for validation.

In [ ]:
class SyntheticRegressionData(d2l.DataModule):
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        self.X, self.y = d2l.synthetic_data(w, b, noise, num_train + num_val)

### A Loader for Every Epoch

What should `get_dataloader` return? The generator `minibatches` of
that section would not do. The trainer asks a `DataModule`
for its loaders once and then iterates over them in every epoch, and a
generator serves one pass and then nothing. The trainer would train for one
epoch and idle through the rest, without an error.

We need a loader that can be iterated in every epoch, draws a fresh random
order on every pass, and reports its number of minibatches through `len`.
The loaders that deep learning frameworks provide do more. For data that do
not fit in memory, or that must be decoded before use, such as large
collections of images, they read examples from disk and prepare the next
minibatches in parallel workers while the model trains on the current one.
Our dataset fits in memory, so none of this is needed yet.

For data held in memory as tensors, we add the method `get_tensorloader` to
`DataModule`. It returns the framework's loader where one suits such data,
and otherwise a short class of our own. The validation examples are read in
their stored order, since we only average the loss over them.

`from_tensor_slices` pairs the tensors example by example, and `batch`
groups them into minibatches. In between, `shuffle` draws each next example
from a buffer: one that holds all training examples gives a new uniformly
random order on every pass, and one of a single example keeps the stored
order.

In [ ]:
@d2l.add_to_class(d2l.DataModule)
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    shuffle_buffer = tensors[0].shape[0] if train else 1
    return tf.data.Dataset.from_tensor_slices(tensors).shuffle(
        buffer_size=shuffle_buffer).batch(self.batch_size)

`get_tensorloader` keeps the examples selected by `indices` and shuffles
them only if `train` is true. `SyntheticRegressionData` passes it the slice
of each split.

In [ ]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

We create the dataset of that section and repeat
the test of the generator: one pass over the training loader, then a second
pass over the same loader.

In [ ]:
data = SyntheticRegressionData(w=d2l.tensor([2, -3.4]), b=4.2)

In [ ]:
loader = data.train_dataloader()
sizes = [len(y) for X, y in loader]
print(f'{len(sizes)} minibatches, the last of size {sizes[-1]}')
print(f'second pass: {len([y for X, y in loader])} minibatches')
for _ in range(2):  # Each new pass starts in a new random order
    X, y = next(iter(loader))
    print('a new pass starts with labels', d2l.numpy(y[:3, 0]).round(2))

The first pass yields 32 minibatches, the last of 8 examples, as the
generator did. This time, however, the second pass yields all 32 minibatches
again, and every pass starts with different examples because the order is
drawn anew. The loader also reports its number of minibatches without a
pass.

In [ ]:
len(loader)

## Training

### The Training Loop and the History

The `Trainer` class fits the parameters of a `Module` to the data of a
`DataModule`. Its constructor takes the number of epochs, `max_epochs`. (The
other two arguments can be ignored until they are needed: `num_gpus` in
that section, and `gradient_clip_val`, which limits the size of
large gradients, in that section.) It also creates a progress
board, on which the trainer plots the losses; we define it below. Training
begins with three preparations: `prepare_data` asks the `DataModule` for its
loaders, `prepare_model` stores the model, and `prepare_optimizer` calls the
model's `configure_optimizers`. Each is a line or two, and a method of its
own so that later sections can override it.

In [ ]:
class Trainer(d2l.HyperParameters):
    """Fit a Module to a DataModule, keeping a per-epoch history."""
    def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
        self.save_hyperparameters()
        self.board = ProgressBoard(xlabel='epoch', ylabel='loss')

    def prepare_data(self, data):
        self.train_dataloader = data.train_dataloader()
        self.val_dataloader = data.val_dataloader()

    def prepare_model(self, model):
        self.model = model

    def prepare_optimizer(self, model):
        self.optim = model.configure_optimizers()

The method `fit(model, data)` is the outer loop of the pseudocode at the
start of this section. After the three preparations it runs `max_epochs`
epochs. In each, `fit_epoch` updates the parameters once per minibatch and
returns the mean of every quantity that `training_step` reports, and
`validate` returns the means over the validation data. The method `record`
keeps these means.

In [ ]:
@d2l.add_to_class(Trainer)
def fit(self, model, data):
    self.prepare_data(data)
    self.prepare_model(model)
    self.prepare_optimizer(model)
    self.history = collections.defaultdict(list)
    self.board.clear()
    self.board.xlim = [0, self.max_epochs]
    for self.epoch in range(self.max_epochs):
        # The training mean averages over the epoch, so it sits mid-epoch
        self.record('train', self.fit_epoch(), x=self.epoch + 0.5)
        if self.val_dataloader is not None:
            self.record('val', self.validate(), x=self.epoch + 1)
        self.board.show(final=False)
    self.board.show()

@d2l.add_to_class(Trainer)
def record(self, split, metrics, x):
    for name in sorted(metrics, key=lambda name: (name != 'loss', name)):
        self.history[f'{split}_{name}'].append(metrics[name])
        self.board.draw(x, metrics[name], f'{split}_{name}')

`fit` stores the epoch in `self.epoch`, so that the methods it calls can
read it, and skips validation when `val_dataloader` returns `None`.

What is left when training ends? `fit` itself returns nothing. Everything is
in `trainer.history`, a dictionary of lists to which `record` appends each
mean. Its keys combine the split with the name of the quantity:
`'train_loss'`, `'val_loss'`, and, for a classifier, `'val_acc'`. Thus
`trainer.history['val_loss'][-1]` is the validation loss after the last
epoch. (`record` puts the loss first, so that a curve keeps its color on the
board from one model to the next.)

### Epoch Means

The means that `fit_epoch` and `validate` return are averages over
examples, not over minibatches: each minibatch counts in proportion to its
size. This matters for the last minibatch of a pass, which is smaller
whenever the minibatch size does not divide the number of examples: 1,000
examples in minibatches of 32 make 31 full minibatches and one of 8, and
with equal weights per minibatch each of the last 8 examples would count
four times as much as the others. The trainer therefore multiplies each
value by the number of examples in its minibatch. This requires a step to
return a mean and never a sum: a loss summed over a minibatch of 32 examples
would enter the average 32 times too large.

`Averager` computes these means. It weights each minibatch by its size, and
it keeps the sums on the device, converting them into Python numbers only
once per epoch, in `mean`, since every such conversion makes Python wait
until a GPU has finished the work queued before it
(that section). With three parameters on a CPU this
saves nothing, and `Averager` may be read as a slightly elaborate way of
taking a mean.

In [ ]:
class Averager:
    """Example-weighted means of batch metrics, summed on the device."""
    def __init__(self):
        self.sums, self.n = {}, 0

    def add(self, metrics, n):
        for name, value in metrics.items():
            self.sums[name] = self.sums.get(name, 0) + value * n
        self.n += n

    def mean(self):
        """One host synchronization per metric; the loss comes first."""
        names = sorted(self.sums, key=lambda name: (name != 'loss', name))
        return {name: float(self.sums[name]) / self.n for name in names}

### Training and Validation Passes

The two passes, `fit_epoch` and `validate`, are the inner loop of `train`
in that section, written against the classes: the
minibatches come from the loader, the loss from the model's steps, and the
update from the optimizer. Both wrap what a step returns with `as_metrics`
and average it with `Averager`. Each minibatch first passes through
`prepare_batch`, a hook that a later chapter overrides to move minibatches
to a GPU; here it returns the minibatch unchanged.

In [ ]:
@d2l.add_to_class(d2l.Trainer)
def prepare_batch(self, batch):
    return batch

The loop that trains every model in the book is longer than linear
regression needs. Its algorithm is the step of that section:
forward pass and loss, backward pass, update, and a clearing of gradients
where the framework keeps them. The other lines prepare the minibatch,
average its metrics, or are hooks for later chapters, and none of them
changes the result of training here.

A function compiled with `tf.function` runs much faster than the same
operations issued one at a time from Python, so this `prepare_model`, which
replaces the one above, compiles the two steps once per call of `fit`. It
first runs one forward pass, because Keras creates a layer's variables only
when the layer first sees data.

In [ ]:
@d2l.add_to_class(d2l.Trainer)
def prepare_model(self, model):
    self.model = model
    loader = (self.train_dataloader if self.train_dataloader is not None
              else self.val_dataloader)  # `evaluate` has no training data
    batch = self.prepare_batch(next(iter(loader)))
    model(*batch[:-1], training=False)  # Keras creates the variables here
    self._train_step = tf.function(self.train_step, reduce_retracing=True)
    self._val_step = tf.function(self.val_step, reduce_retracing=True)

@d2l.add_to_class(d2l.Trainer)
def train_step(self, batch):
    with tf.GradientTape() as tape:
        out = d2l.as_metrics(self.model.training_step(batch))
    params = self.model.trainable_variables
    grads = tape.gradient(out['loss'], params)
    if self.gradient_clip_val > 0:
        grads = self.clip_gradients(self.gradient_clip_val, grads)
    self.optim.apply_gradients(zip(grads, params))
    return out

@d2l.add_to_class(d2l.Trainer)
def val_step(self, batch):
    return d2l.as_metrics(self.model.validation_step(batch))

@d2l.add_to_class(d2l.Trainer)
def fit_epoch(self):
    metrics = d2l.Averager()
    for batch in map(self.prepare_batch, self.train_dataloader):
        metrics.add(self._train_step(batch), len(batch[-1]))
    return metrics.mean()

@d2l.add_to_class(d2l.Trainer)
def validate(self):
    metrics = d2l.Averager()
    for batch in map(self.prepare_batch, self.val_dataloader):
        metrics.add(self._val_step(batch), len(batch[-1]))
    return metrics.mean()

### The Progress Board

`record` also adds each mean to the progress board, which `fit` redraws
after every epoch so that we can watch training in a notebook. The class
`ProgressBoard` keeps one list of points per curve: `draw(x, y, label)`
appends the point `(x, y)` to the curve named `label`, `show` redraws all
curves, and `clear` forgets them. The drawing code has nothing to teach us
here and is in that section. The board can be configured before
training, for instance with `trainer.board.yscale = 'log'` for a logarithmic
loss axis, or with `trainer.board.display = False` to train without
plotting.

Before we read our first pair of training curves, note that they are
computed with different parameters. The validation mean uses the parameters
at the end of the epoch, while the training mean averages over minibatches
seen while the parameters changed. `fit` therefore draws the training mean
of each epoch at its middle, at 0.5, 1.5, and so on, and the validation mean
at its end, at 1, 2, and so on.

In [ ]:
class ProgressBoard(d2l.HyperParameters):
    """Curves that grow during training; `show` redraws them."""
    def __init__(self, xlabel=None, ylabel=None, xlim=None, ylim=None,
                 xscale='linear', yscale='linear', figsize=(3.5, 2.5),
                 display=True):
        self.save_hyperparameters()
        self.clear()

    def clear(self):
        """Forget all curves; the next `show` starts a new figure."""
        self.data, self.handle = {}, None

    def draw(self, x, y, label):
        """Append the point (x, y) to the curve called `label`."""
        xs, ys = self.data.setdefault(label, ([], []))
        xs.append(x)
        ys.append(y)

    def show(self, final=True):
        raise NotImplementedError

## Training Linear Regression Again

We now train the model of that section once more, through
the classes, on the dataset `data` created above. The arguments of `train`
have moved to the classes that own them: the learning rate 0.03 to the
model, the minibatch size 32 to the data (it is the default), and the ten
epochs to the trainer. We use `d2l.Trainer`, the class of this section
together with the drawing code of that section.

In [ ]:
model = LinearRegressionScratch(2, lr=0.03)
trainer = d2l.Trainer(max_epochs=10)
trainer.board.yscale = 'log'
trainer.fit(model, data)

The validation loss behaves as in that section: it falls by
a factor of about 7 per epoch and levels off near the floor of
$5 \times 10^{-5}$. The training curve lies on top of it. With three
parameters and 1,000 examples, the model can fit almost none of the noise
in its training labels, so it does as well on new examples as on the old
ones. Has anything changed besides the organization of the code? The final
validation loss and the parameter errors answer that.

In [ ]:
w_error = d2l.numpy(d2l.reshape(model.w, data.w.shape) - data.w)
b_error = d2l.numpy(model.b - data.b)
print(f"validation loss: {trainer.history['val_loss'][-1]:.1e}")
print('error in w:', ', '.join(f'{e:.1e}' for e in w_error))
print('error in b:', ', '.join(f'{e:.1e}' for e in b_error))

Nothing has. The validation loss is at the floor, and the parameters are
within about $10^{-3}$ of the true ones, typically a few times $10^{-4}$, as
in that section. For a single model the classes would not
be worth their extra lines. The next section is the first to need them: it
replaces the model's parameters, loss, and optimizer with framework
components and trains the result with this trainer, unchanged.

## Summary

Every model in this book is trained by the same loop, so we write it once,
as a `Trainer`, and keep apart what changes between models. A `Module`
holds the parameters, the forward pass, the loss, and the optimizer; a
`DataModule` serves training and validation data from loaders that can be
iterated in every epoch. A model's steps return minibatch means and do
nothing else, so any loop can train or evaluate the model. The trainer
averages these means over the examples of each epoch, weighting each
minibatch by its size, and keeps the results in `trainer.history`.
Rewritten in this form, the linear regression model of
that section reaches the loss floor and recovers the true
parameters as before, and a change to training is now made in one place.

## Exercises

1. [code] **Methods added later.** Register a second method `do` on class
   `A` with a different body, predict which version `a.do()` runs, and
   check. Then attach a function `greet(self)` to the instance instead, with
   `setattr(a, 'greet', greet)`, call `a.greet()`, and explain the error.

1. **Optimizer and model.** The `Module` class keeps the optimizer in
   `configure_optimizers`, a *method of the model*, rather than passing it
   as an argument to `Trainer`. State the advantages of this design choice,
   and describe a training setup in which putting the optimizer on the
   model becomes awkward.

1. **Reading the training curves.** The trainer draws the mean training
   loss of each epoch at the middle of the epoch and the validation loss
   at its end.
    1. Early in training, the training curve often lies above the
       validation curve, although both sets of examples come from the same
       distribution. Explain why from the way the two means are computed.
    1. Suppose `fit` computed the training loss like the validation loss,
       with the parameters at the end of each epoch. What would this cost
       per epoch, and what would the gap between the two curves then
       measure?

1. [code] **Early stopping.** Training for a fixed number of epochs may
   stop too early or too late. Write a subclass of `d2l.Trainer` whose
   `fit` ends training once the validation loss has not decreased for
   `patience` consecutive epochs, deciding from `self.history` alone.
   Explain why no model or dataset class has to change. Test your trainer
   on the synthetic regression data with a large `max_epochs`.

1. [code] **Test split.** Write a subclass of `SyntheticRegressionData`
   whose constructor also draws a test set from the same distribution and
   whose method `test_dataloader` serves it. Train a model on the original
   data, and compute its mean loss on the test set once, after the last
   epoch, with the model's `validation_step`. Explain why the test set is
   evaluated once at the end rather than after every epoch like the
   validation data, and what would go wrong if the epoch with the best test
   loss were reported. that section returns to this
   question.

1. [code] **Planck's law.** ● The same `Module` and `Trainer` fit any model
   whose loss can be differentiated, not only a linear one. Use
   [Planck's law](https://en.wikipedia.org/wiki/Planck%27s_law) to
   determine the temperature of a black body from measurements of its
   spectral radiance at several wavelengths. The spectral radiance is

    $$B(\lambda, T) = \frac{2 hc^2}{\lambda^5} \cdot \left(\exp \frac{h c}{\lambda k T} - 1\right)^{-1},$$

    where $\lambda$ is the wavelength, $T$ is the temperature, $c$ is the
    speed of light, $h$ is Planck's constant, and $k$ is the Boltzmann
    constant. Generate measurements at a known $T$, then write a
    `d2l.Module` subclass whose `forward` computes $B(\lambda, T)$ with $T$
    as its only parameter and fit it with `d2l.Trainer`. Choose units in
    which $B$ and $T$ are of order one before fitting.

7. **A partial last minibatch.** With 1,000 examples per split and
   `batch_size=32`, each loader ends a pass with a partial minibatch of 8
   examples. `Dataset.batch` discards it when given `drop_remainder=True`.
   The training loader reshuffles every epoch, whereas the validation loader
   reads in a fixed order. For each loader, state which examples a discarded
   partial minibatch leaves out over many epochs, and decide for which
   loader discarding is acceptable.

8. [code] **On-the-fly generation.** Write a `DataModule` subclass that
   stores no $\mathbf{X}$ or $\mathbf{y}$. Its training loader instead draws
   every minibatch fresh from the data-generating process when the
   minibatch is requested.
    1. A stream of fresh minibatches has no natural end, yet the `Trainer`
       ends an epoch only when a pass over the training loader ends. Decide
       how many minibatches one epoch yields, make every pass stop there,
       and make `len` return that number.
    1. Decide whether the validation loader should also draw fresh examples
       on every pass, and implement your choice. Justify it by what the
       validation loss is used for.

[Discussions](https://d2l.discourse.group/t/6647)